# CNN Assignment: Fashion-MNIST Classification

Use the same CNN architecture from the MNIST tutorial to train, evaluate, and analyze a model on the **Fashion-MNIST** dataset (10 categories of clothing instead of digits).

Designed to run in **Google Colab**.
> Optional: In Colab, go to `Runtime > Change runtime type` and select **GPU** to train faster.

## Instructions
- Some sections are marked "(provided)" — you don't need to change those.
- For each question, fill in the empty code cell (or the missing `TODO` part) below it.
- Keep the "Function Notes" cells — they list the functions you'll need.
- Run cells from top to bottom.

## Setup: imports, device, and seed (provided)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## Load Fashion-MNIST (provided)

Fashion-MNIST is a drop-in replacement for MNIST: same image size (28x28 grayscale) and same number of classes (10), but the classes are clothing items instead of digits.

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

print("Training set size:", len(train_dataset))
print("Test set size:", len(test_dataset))

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    img, lbl = train_dataset[i]
    axes[i].imshow(img.squeeze(), cmap="gray")
    axes[i].set_title(class_names[lbl])
    axes[i].axis("off")
plt.show()

## CNN architecture (provided — identical to the MNIST tutorial)

```
Input image: 1 x 28 x 28
Conv2d(1 -> 16, 3x3, padding=1) -> ReLU -> MaxPool(2x2)   => 16 x 14 x 14
Conv2d(16 -> 32, 3x3, padding=1) -> ReLU -> MaxPool(2x2)  => 32 x 7 x 7
Flatten                                                    => 1,568
Linear(1,568 -> 128) -> ReLU -> Dropout
Linear(128 -> 10)
```

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### Question 4.1 — Split into training and validation sets

`val_fraction`, `val_size`, and `train_size` are already computed below using a 90% / 10% split. Use `random_split` to create `train_subset` and `val_subset` from `train_dataset`, then print the size of all three sets.

## Function Notes
- `random_split(dataset, lengths, generator)` — randomly splits a dataset into subsets of the given sizes.
- `torch.Generator().manual_seed(seed)` — makes the split reproducible.

In [ ]:
# provided
val_fraction = 0.1
val_size = int(len(train_dataset) * val_fraction)
train_size = len(train_dataset) - val_size

In [ ]:
train_subset, val_subset = ???

## Train a baseline model (provided)

This builds the `DataLoader`s and the `train_one_epoch`/`evaluate` helper functions, then trains a model for `num_epochs` epochs, recording the loss/accuracy history — the same pattern used in the tutorial. No changes needed here.

In [ ]:
# provided
batch_size = 64
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# provided
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()

    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct / len(loader.dataset)
    return avg_loss, accuracy


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()

    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct / len(loader.dataset)
    return avg_loss, accuracy


model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 12
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}/{num_epochs} | "
          f"train loss: {train_loss:.4f}, train acc: {train_acc:.4f} | "
          f"val loss: {val_loss:.4f}, val acc: {val_acc:.4f}")

### Question 4.2 — Save the best model

Below is the same training loop as above, but retraining a fresh model. Fill in the missing `TODO` part so that whenever `val_loss` improves, the model's weights are saved to `checkpoint_path`.

## Function Notes
- `model.state_dict()` — a dictionary containing the model's current weights.
- `torch.save(object, path)` — saves a Python object to a file.

In [ ]:
# provided (except the TODO block)
model = SimpleCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

checkpoint_path = "best_fashion_model.pth"
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}/{num_epochs} | "
          f"train loss: {train_loss:.4f}, train acc: {train_acc:.4f} | "
          f"val loss: {val_loss:.4f}, val acc: {val_acc:.4f}")

    # TODO: if val_loss improved over best_val_loss, save model.state_dict()
    # to checkpoint_path and update best_val_loss
    if ??? :
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  -> New best model saved (val loss: {val_loss:.4f})")

### Question 4.3 — Plot learning curves (Provided)

Using the history you recorded above, plot training vs. validation loss, and training vs. validation accuracy, across epochs.

## Function Notes
- `plt.subplots(rows, cols, figsize=(w, h))` — creates a figure with multiple subplots.
- `ax.plot(x, y, label=...)` — draws a line chart.
- `ax.set_xlabel(text)` / `ax.set_ylabel(text)` — labels an axis.
- `ax.legend()` — shows a legend using the `label=` values.

In [ ]:
# provided
epochs_range = range(1, num_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], label="Train loss")
axes[0].plot(epochs_range, history["val_loss"], label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss vs. epoch")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="Train accuracy")
axes[1].plot(epochs_range, history["val_acc"], label="Val accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy vs. epoch")
axes[1].legend()

plt.show()

### Question 4.4 — Identify the point where overfitting occurs

Looking at your learning curves from Question 4.3, find the epoch where overfitting occurs. Report that epoch number, and briefly explain how you found it.

**Your answer:**

_(Write the epoch number and your explanation here.)_

### Question 4.5 — Continue training to 30 epochs

Continue training the model from Question 4.2 for more epochs, until it reaches 30 total. Keep saving the best checkpoint. Then check whether validation performance improved further, and explain why or why not (relate this to your answer in Question 4.4).

In [ ]:
# provided (except the TODO block)
for epoch in range(num_epochs + 1, 30 + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}/30 | "
          f"train loss: {train_loss:.4f}, train acc: {train_acc:.4f} | "
          f"val loss: {val_loss:.4f}, val acc: {val_acc:.4f}")

    # TODO: if val_loss improved over best_val_loss, save the checkpoint again
    if ??? :
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  -> New best model saved (val loss: {val_loss:.4f})")

num_epochs = 30
epochs_range = range(1, num_epochs + 1)

In [ ]:
# provided
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], label="Train loss")
axes[0].plot(epochs_range, history["val_loss"], label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss vs. epoch (30 epochs)")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="Train accuracy")
axes[1].plot(epochs_range, history["val_acc"], label="Val accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy vs. epoch (30 epochs)")
axes[1].legend()

plt.show()

**Your answer:**

_(Did validation performance improve further after training to 30 epochs? Explain why or why not, relating this to your answer in Question 4.4.)_

### Question 4.6 — Load the best model (Provided)

Create a new instance of `SimpleCNN` and load the weights saved in Question 4.2 into it.

## Function Notes
- `torch.load(path, map_location)` — loads a saved object from disk.
- `model.load_state_dict(state_dict)` — copies loaded weights into a model instance.

In [ ]:
loaded_model = SimpleCNN().to(device)
loaded_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
loaded_model.eval()
print("Loaded best model from", checkpoint_path)

### Question 4.7 — Test on the test set (Provided)

Evaluate the loaded best model on `test_dataset` (the untouched test set). Report the test loss and test accuracy.

## Function Notes
- Reuse (or rewrite) an `evaluate`-style function: set `model.eval()`, loop over a `DataLoader` inside `torch.no_grad()`, and average the loss/accuracy over the dataset.

In [ ]:
test_loss, test_acc = evaluate(loaded_model, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")